## Template TICKER analysis

### 0. Setup and readers

In [1]:
TICKER = "XMR"
TICK = 0.001

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ноутбук лежит глубже BacktestingBasics — добавляем research/ в путь, чтобы найти пакет tools.
repo_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                 if (p / ".git").exists())
research_dir = repo_root / "research"
if str(research_dir) not in sys.path:
    sys.path.insert(0, str(research_dir))

from tools import ROOT
from tools.readers.lighter import LobReader, TradesReader

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

In [3]:
import plotly.io as pio

pio.renderers.default = "iframe"  # рендер plotly внутри JupyterLab (иначе вывод пустой)

In [4]:
def style_figure(fig, height):
    """Единое оформление для всех графиков ноутбука."""
    fig.update_layout(template="plotly_white", height=height, showlegend=False,
                      bargap=0.02, margin=dict(l=55, r=20, t=50, b=45))
    return fig


def add_percentile_band(fig, row, x, low, median, high, rgb="76,120,168"):
    """Линия медианы и затенённая полоса [low, high] в подграфике (row, 1)."""
    fig.add_trace(go.Scatter(x=x, y=high, mode="lines", line_width=0, hoverinfo="skip"), row, 1)
    fig.add_trace(go.Scatter(x=x, y=low, mode="lines", line_width=0, fill="tonexty",
                             fillcolor=f"rgba({rgb},0.2)", hoverinfo="skip"), row, 1)
    fig.add_trace(go.Scatter(x=x, y=median, mode="lines",
                             line=dict(color=f"rgb({rgb})", width=1.2)), row, 1)


def percentile_summary(series, decimals):
    """Перцентили + mean/max одной серии как dict — кирпичик строки-артефакта."""
    levels = (1, 5, 25, 50, 75, 90, 95, 99)
    summary = {f"p{level}": np.percentile(series, level) for level in levels}
    summary["mean"] = series.mean()
    summary["max"] = series.max()
    return {name: round(float(value), decimals) for name, value in summary.items()}


def build_taker_orders(trades):
    """Тейкер-филлы (type==trade, без ликвидаций) и их агрегация в тейкер-ордера.
    Общее ядро для D (свипы) и B (adverse selection). Возвращает (taker_fills, orders).
    Несём оба движковых времени: t_match (timestamp, матч) и t_commit (transaction_time)."""
    maker_ask = trades["is_maker_ask"].to_numpy()
    is_trade = trades["type"].eq("trade").to_numpy()
    taker_fills = pd.DataFrame({
        "order_id": np.where(maker_ask, trades["bid_id"], trades["ask_id"]),
        "account": np.where(maker_ask, trades["bid_account_id"], trades["ask_account_id"]),
        "buy": maker_ask,
        "price": trades["price"].to_numpy(),
        "notional": (trades["price"] * trades["size"]).to_numpy(),
        "block": trades["block_height"].to_numpy(),
        "t_match": trades["timestamp"].to_numpy(),
        "t_commit": trades["transaction_time"].to_numpy(),
    })[is_trade]

    orders = taker_fills.groupby("order_id").agg(
        t_match=("t_match", "min"),
        t_commit=("t_commit", "min"),
        notional=("notional", "sum"),
        n_legs=("price", "count"),
        n_levels=("price", "nunique"),
        px_min=("price", "min"),
        px_max=("price", "max"),
        px_mean=("price", "mean"),
        block_span=("block", lambda block: block.max() - block.min()),
        buy=("buy", "first"),
    )
    orders["category"] = np.where(orders["block_span"] > 1, "algo_twap",
                                  np.where(orders["n_levels"] >= 2, "sweep", "single_level"))
    orders["depth_bps"] = (orders["px_max"] - orders["px_min"]) / orders["px_mean"] * 1e4
    return taker_fills, orders


def drift_matrix(anchor_us, side, book_ts, book_mid, book_valid, horizons_s):
    """Подписанный дрифт mid по сетке горизонтов относительно якоря anchor_us (µs).
    m₀ = последний снапшот строго до anchor; m_h = последний с временем ≤ anchor+h.
    Возвращает (drift[events×horizons] bps, reach bool, i0 clipped, valid0)."""
    i0 = np.searchsorted(book_ts, anchor_us, side="left") - 1
    valid0 = (i0 >= 0)
    i0 = np.clip(i0, 0, None)
    valid0 = valid0 & book_valid[i0]
    mid0 = book_mid[i0]

    drift = np.full((len(anchor_us), len(horizons_s)), np.nan)
    reach = np.zeros_like(drift, dtype=bool)
    for j, h in enumerate(horizons_s):
        ih = np.clip(np.searchsorted(book_ts, anchor_us + int(h * 1e6), side="right") - 1, 0, len(book_ts) - 1)
        ok = valid0 & (anchor_us + h * 1e6 <= book_ts[-1]) & book_valid[ih]
        drift[:, j] = np.where(ok, side * (book_mid[ih] - mid0) / mid0 * 1e4, np.nan)
        reach[:, j] = ok
    return drift, reach, i0, valid0

In [5]:
# Покрытие по часам у символов разное — берём все доступные файлы (сортировка = хронология).
lob_paths = sorted((ROOT / f"data/lighter/{TICKER}/lob").glob("*/*.parquet"))
trades_paths = sorted((ROOT / f"data/lighter/{TICKER}/trades").glob("*/*.jsonl.zst"))

print(f"{TICKER}: {len(lob_paths)} lob files, {len(trades_paths)} trade files")
print(f"  {lob_paths[0].relative_to(ROOT)} ... {lob_paths[-1].relative_to(ROOT)}")

XMR: 20 lob files, 20 trade files
  data/lighter/XMR/lob/20260711/22.parquet ... data/lighter/XMR/lob/20260712/17.parquet


In [6]:
lobs = LobReader(lob_paths).load()

span_min = (lobs["recv_ts"].iloc[-1] - lobs["recv_ts"].iloc[0]) / 1e6 / 60
print(f"lob snapshots: {len(lobs):,} | span {span_min:.1f} min "
      f"| recv_ts monotonic: {lobs['recv_ts'].is_monotonic_increasing}")
lobs.head()

lob snapshots: 237,765 | span 1167.9 min | recv_ts monotonic: True


,sent_ts,recv_ts,ask_px_1,ask_sz_1,ask_px_2,ask_sz_2,ask_px_3,ask_sz_3,ask_px_4,ask_sz_4,ask_px_5,ask_sz_5,ask_px_6,ask_sz_6,ask_px_7,ask_sz_7,ask_px_8,ask_sz_8,ask_px_9,ask_sz_9,ask_px_10,ask_sz_10,ask_px_11,ask_sz_11,ask_px_12,ask_sz_12,ask_px_13,ask_sz_13,ask_px_14,ask_sz_14,...,bid_px_16,bid_sz_16,bid_px_17,bid_sz_17,bid_px_18,bid_sz_18,bid_px_19,bid_sz_19,bid_px_20,bid_sz_20,bid_px_21,bid_sz_21,bid_px_22,bid_sz_22,bid_px_23,bid_sz_23,bid_px_24,bid_sz_24,bid_px_25,bid_sz_25,bid_px_26,bid_sz_26,bid_px_27,bid_sz_27,bid_px_28,bid_sz_28,bid_px_29,bid_sz_29,bid_px_30,bid_sz_30
0,1783809125596000,1783809125614576,322.347,2.195,322.348,4.77,322.382,4.77,322.384,31.225,322.426,6.361,322.427,4.77,322.431,5.84,322.448,1.858,322.465,12.218,322.515,2.566,322.519,4.77,322.52,5.066,322.54,23.252,322.568,1.859,...,321.907,0.311,321.899,1.861,321.884,25.906,321.864,4.77,321.839,17.077,321.821,25.906,321.811,0.311,321.801,15.902,321.77,15.902,321.763,25.906,321.704,20.174,321.678,25.906,321.651,24.85,321.6,0.311,321.44,86.397
1,1783809125793000,1783809125793617,322.347,2.195,322.348,4.77,322.382,4.77,322.384,31.225,322.426,6.361,322.427,4.77,322.431,5.84,322.448,1.858,322.465,12.218,322.515,2.566,322.519,4.77,322.52,5.066,322.54,23.252,322.568,1.859,...,321.907,0.311,321.899,1.861,321.884,25.906,321.864,4.77,321.839,17.077,321.821,25.906,321.811,0.311,321.801,15.902,321.77,15.902,321.763,25.906,321.704,20.174,321.678,25.906,321.651,24.85,321.6,0.311,321.44,86.397
2,1783809125842000,1783809125842941,322.347,2.195,322.348,4.77,322.382,4.77,322.384,31.225,322.426,6.361,322.427,4.77,322.431,5.84,322.448,1.858,322.462,12.218,322.515,2.566,322.519,4.77,322.52,5.066,322.54,23.252,322.568,1.859,...,321.907,0.311,321.899,1.861,321.884,25.906,321.864,4.77,321.839,17.077,321.821,25.906,321.811,0.311,321.801,15.902,321.77,15.902,321.763,25.906,321.704,20.174,321.678,25.906,321.651,24.85,321.6,0.311,321.44,86.397
3,1783809125895000,1783809125896363,322.347,2.195,322.348,4.77,322.382,4.77,322.384,31.225,322.426,6.361,322.427,4.77,322.431,5.84,322.448,1.858,322.462,12.218,322.515,2.566,322.519,4.77,322.52,5.066,322.54,23.252,322.568,1.859,...,321.907,0.311,321.899,1.861,321.884,25.906,321.864,4.77,321.839,17.077,321.821,25.906,321.811,0.311,321.801,15.902,321.77,15.902,321.763,25.906,321.704,20.174,321.678,25.906,321.651,24.85,321.6,0.311,321.44,86.397
4,1783809125943000,1783809125944436,322.347,2.195,322.348,4.77,322.382,4.77,322.384,31.225,322.426,6.361,322.427,4.77,322.431,5.84,322.448,1.858,322.462,12.218,322.515,2.566,322.519,4.77,322.52,5.066,322.54,23.252,322.568,1.859,...,321.907,0.311,321.899,1.861,321.884,25.906,321.864,4.77,321.839,17.077,321.821,25.906,321.811,0.311,321.801,15.902,321.77,15.902,321.763,25.906,321.704,20.174,321.678,25.906,321.651,24.85,321.6,0.311,321.44,86.397


In [7]:
# Sanity: вбитый вручную TICK совпадает с фактическим шагом ценовой сетки (ловит опечатку).
grid_prices = lobs[[f"{s}_px_{i}" for s in ("ask", "bid") for i in range(1, 31)]].to_numpy().ravel()
grid_prices = grid_prices[grid_prices > 0]
price_grid = int(np.gcd.reduce(np.diff(np.unique(np.round(grid_prices * 1e8).astype(np.int64))))) / 1e8
assert abs(price_grid - TICK) < 1e-12, f"{TICKER}: TICK={TICK} != price grid {price_grid} — поправь TICK"
print(f"TICK {TICK} confirmed vs price grid")

TICK 0.001 confirmed vs price grid


In [8]:
trades = TradesReader(trades_paths).load()

# Ридер сохраняет порядок поступления; внутри батча сделки не отсортированы по времени.
print(f"trades: {len(trades):,} fills | timestamp monotonic: {trades['timestamp'].is_monotonic_increasing}")
trades.head()

trades: 510 fills | timestamp monotonic: False


,receive_timestamp,timestamp,transaction_time,trade_id,type,price,size,is_maker_ask,ask_account_id,bid_account_id,ask_id,bid_id,taker_fee,maker_fee,taker_position_size_before,maker_position_size_before,block_height
0,1783809125629103,1783807684459000,1783807684755408,25123760822,trade,322.499,0.058,True,314236,722573,21955048350346991,22236523018953979,273,28,-0.186,-9.567,289076181
1,1783809125629103,1783807578019000,1783807578099465,25123708967,trade,322.554,0.116,True,418682,218734,21955048350346061,22236523018954107,0,36,-0.116,-1030.200,289075563
2,1783809125629103,1783807438183000,1783807438317380,25123629141,trade,322.135,0.093,False,511297,314236,21955048350346455,22236523018954337,0,28,0.093,-9.660,289074593
3,1783809125629103,1783807297141000,1783807297194927,25123555467,trade,322.288,1.171,False,314236,418682,21955048350345997,22236523018954824,196,36,-8.489,-1031.371,289073684
4,1783809125629103,1783807296784000,1783807296839906,25123555368,trade,322.288,0.859,False,728846,418682,21955048350345972,22236523018954824,280,36,0.859,-1032.230,289073682


### 1. Spread analysis

In [9]:
# Спред из top-of-book. Валидность книги: обе стороны есть и не скрещены.
best_ask = lobs["ask_px_1"].to_numpy()
best_bid = lobs["bid_px_1"].to_numpy()
valid_book = (best_ask > 0) & (best_bid > 0) & (best_ask >= best_bid)

mid = (best_ask[valid_book] + best_bid[valid_book]) / 2
abs_spread = best_ask[valid_book] - best_bid[valid_book]
ref_price = float(np.median(mid))

spread = pd.DataFrame({
    "recv_ts": lobs["recv_ts"].to_numpy()[valid_book],
    "ticks": abs_spread / TICK,
    "bps": abs_spread / mid * 1e4,
})

print(f"valid snapshots: {valid_book.sum():,} / {len(lobs):,} (dropped {(~valid_book).sum()})")
spread[["ticks", "bps"]].describe(percentiles=[.5, .9]).round(3)

valid snapshots: 237,765 / 237,765 (dropped 0)


,ticks,bps
count,237765.000,237765.000
mean,262.678,8.028
std,64.142,1.958
min,1.000,0.030
50%,270.000,8.259
90%,332.000,10.154
max,993.000,30.373


In [10]:
# Распределение спреда: тики слева, bps справа. Линии p50 (сплошная) и p90 (пунктир).
hist = make_subplots(rows=1, cols=2,
                     subplot_titles=("Spread distribution, ticks", "Spread distribution, bps"))
for unit, col in (("ticks", 1), ("bps", 2)):
    hist.add_trace(go.Histogram(x=spread[unit], nbinsx=80, marker_color="#4C78A8"), 1, col)
    for level, dash in ((50, "solid"), (90, "dot")):
        hist.add_vline(x=float(np.percentile(spread[unit], level)),
                       line=dict(color="#E45756", dash=dash, width=1), row=1, col=col)
    hist.update_xaxes(range=[0, float(np.percentile(spread[unit], 99))], title_text=unit, row=1, col=col)

style_figure(hist, 380).show()

In [11]:
# Спред по сессии: медиана в бинах + полоса p10–p90 (тики сверху, bps снизу).
BIN_SECONDS = 30
start_ts = int(spread["recv_ts"].iloc[0])
bin_index = (spread["recv_ts"] - start_ts) // (BIN_SECONDS * 1_000_000)
binned = spread.groupby(bin_index)

session = pd.DataFrame({
    "minute": binned["recv_ts"].first().sub(start_ts).div(6e7),
    "ticks_p10": binned["ticks"].quantile(.1),
    "ticks_p50": binned["ticks"].median(),
    "ticks_p90": binned["ticks"].quantile(.9),
    "bps_p10": binned["bps"].quantile(.1),
    "bps_p50": binned["bps"].median(),
    "bps_p90": binned["bps"].quantile(.9),
})

session_fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                            subplot_titles=("Spread over session, ticks", "Spread over session, bps"))
add_percentile_band(session_fig, 1, session["minute"],
                    session["ticks_p10"], session["ticks_p50"], session["ticks_p90"])
add_percentile_band(session_fig, 2, session["minute"],
                    session["bps_p10"], session["bps_p50"], session["bps_p90"])
style_figure(session_fig, 560)
session_fig.update_xaxes(title_text="minutes from session start", row=2, col=1)
session_fig.update_yaxes(title_text="ticks", row=1, col=1)
session_fig.update_yaxes(title_text="bps", row=2, col=1)
session_fig.show()

In [12]:
# Строка-артефакт для сводной таблицы по всем символам.
spread_shares = {
    "share_spread_eq_1_tick": (spread["ticks"] == 1).mean(),
    "share_spread_le_2_ticks": (spread["ticks"] <= 2).mean(),
    "share_spread_ge_5_ticks": (spread["ticks"] >= 5).mean(),
}

A_row = pd.Series({
    "ticker": TICKER,
    "tick": TICK,
    "bps_per_tick": round(TICK / ref_price * 1e4, 4),
    "n_snapshots": int(valid_book.sum()),
    **{f"spread_ticks_{name}": value for name, value in percentile_summary(spread["ticks"], 1).items()},
    **{f"spread_bps_{name}": value for name, value in percentile_summary(spread["bps"], 3).items()},
    **{name: round(float(value), 4) for name, value in spread_shares.items()},
}, name=TICKER)
A_row

ticker                        XMR
tick                        0.001
bps_per_tick               0.0305
n_snapshots                237765
spread_ticks_p1              49.0
spread_ticks_p5             137.0
spread_ticks_p25            234.0
spread_ticks_p50            270.0
spread_ticks_p75            303.0
spread_ticks_p90            332.0
spread_ticks_p95            349.0
spread_ticks_p99            392.0
spread_ticks_mean           262.7
spread_ticks_max            993.0
spread_bps_p1               1.489
spread_bps_p5                4.17
spread_bps_p25               7.17
spread_bps_p50              8.259
spread_bps_p75              9.252
spread_bps_p90             10.154
spread_bps_p95             10.638
spread_bps_p99             11.972
spread_bps_mean             8.028
spread_bps_max             30.373
share_spread_eq_1_tick        0.0
share_spread_le_2_ticks    0.0017
share_spread_ge_5_ticks     0.998
Name: XMR, dtype: object

### 2. Adverse selection

In [13]:
# События B = удары тейкера (type=trade, без algo/twap). Ядро — общий хелпер (см. секцию 0).
taker_fills, orders = build_taker_orders(trades)
events = orders[orders["category"] != "algo_twap"].copy()   # sweep + single_level = основная выборка
events["side"] = np.where(events["buy"], 1, -1)
algo_twap_share = orders.loc[orders["category"] == "algo_twap", "notional"].sum() / orders["notional"].sum()

print(f"{len(events):,} events (main) | algo/twap share (notional): {algo_twap_share:.4f}")
events[["t_match", "t_commit", "side", "notional", "n_levels", "category"]].head()

421 events (main) | algo/twap share (notional): 0.0000


,t_match,t_commit,side,notional,n_levels,category
order_id,,,,,,
21955048350135092,1783802362515000,1783802362515976,-1,99.25416,1,single_level
21955048350314366,1783800855907000,1783800856046649,-1,2238.43680,1,single_level
21955048350314379,1783800855907000,1783800856101158,-1,167.71664,1,single_level
21955048350314387,1783800856129000,1783800856159102,-1,99.98492,1,single_level
21955048350314388,1783800856169000,1783800856171641,-1,99.98492,1,single_level


In [14]:
# Таймлайн книги для as-of: engine-emission время (sent_ts, ms-квант), mid, валидность.
book_ts = lobs["sent_ts"].to_numpy()
book_ask = lobs["ask_px_1"].to_numpy()
book_bid = lobs["bid_px_1"].to_numpy()
book_mid = (book_ask + book_bid) / 2
book_valid = (book_ask > 0) & (book_bid > 0) & (book_ask >= book_bid)

if not (np.diff(book_ts) >= 0).all():   # as-of требует неубывающего времени
    order_ts = np.argsort(book_ts, kind="stable")
    book_ts, book_ask, book_bid = book_ts[order_ts], book_ask[order_ts], book_bid[order_ts]
    book_mid, book_valid = book_mid[order_ts], book_valid[order_ts]
    print("warning: sent_ts не монотонен — отсортировано")

# Гэп записи (провизорно, уточним кросс-канальной живостью в агрегации): Δsent_ts > адаптивного порога.
step_us = np.diff(book_ts)
gap_threshold_us = max(10_000_000, 20 * np.median(step_us))
gap_at = np.where(step_us > gap_threshold_us)[0]
print(f"book cadence median: {np.median(step_us) / 1000:.1f} ms | "
      f"gap threshold: {gap_threshold_us / 1e6:.1f} s | gaps: {len(gap_at)}")

book cadence median: 101.0 ms | gap threshold: 10.0 s | gaps: 0


In [15]:
# As-of дрифт по обоим якорям вилки:
#   AS_hi — m₀ строго до timestamp (матч): импакт не протекает в m₀ по построению (g≥0). Безопасный.
#   AS_lo — m₀ до transaction_time (коммит): импакт частично протекает → нижняя граница.
HORIZONS_S = [0.1, 0.2, 0.35, 0.5, 1, 2, 10, 30]
side = events["side"].to_numpy()

drift_hi, reach_hi, i0_hi, valid0_hi = drift_matrix(events["t_match"].to_numpy(), side,
                                                    book_ts, book_mid, book_valid, HORIZONS_S)
drift_lo, reach_lo, i0_lo, valid0_lo = drift_matrix(events["t_commit"].to_numpy(), side,
                                                    book_ts, book_mid, book_valid, HORIZONS_S)

mid0 = book_mid[i0_hi]
halfspread_bps0 = np.where(valid0_hi, (book_ask[i0_hi] - book_bid[i0_hi]) / 2 / mid0 * 1e4, np.nan)
drift_hi_df = pd.DataFrame(drift_hi, columns=[f"h_{h}s" for h in HORIZONS_S], index=events.index)

print(f"valid m0 — hi (timestamp): {valid0_hi.sum():,} | lo (transaction_time): {valid0_lo.sum():,} / {len(events):,}")
drift_hi_df[valid0_hi].describe(percentiles=[.5]).round(3)

valid m0 — hi (timestamp): 376 | lo (transaction_time): 376 / 421


,h_0.1s,h_0.2s,h_0.35s,h_0.5s,h_1s,h_2s,h_10s,h_30s
count,376.000,376.000,376.000,376.000,376.000,376.000,375.000,366.000
mean,1.253,2.261,3.088,3.391,3.906,4.434,5.110,5.247
std,3.466,4.142,5.418,5.209,5.823,6.314,7.493,9.286
min,-8.339,-8.339,-25.201,-25.140,-29.957,-29.866,-29.607,-25.887
50%,-0.000,1.047,1.553,2.187,2.869,3.503,3.795,4.315
max,25.838,25.133,29.420,25.185,32.335,32.431,36.162,47.350


In [16]:
# Вилка AS_hi − AS_lo по горизонтам (equal-weighted mean). Для пилота на 3 символах:
# если Δ₃₅₀ < ~0.2 bps везде — батчим 22 только на AS_hi; иначе тащим обе версии.
as_curves = pd.DataFrame({
    "AS_hi_bps": np.nanmean(drift_hi, axis=0),
    "AS_lo_bps": np.nanmean(drift_lo, axis=0),
}, index=[f"{h}s" for h in HORIZONS_S])
as_curves["delta_bps"] = as_curves["AS_hi_bps"] - as_curves["AS_lo_bps"]
as_curves.round(3)

,AS_hi_bps,AS_lo_bps,delta_bps
0.1s,1.253,1.930,-0.677
0.2s,2.261,2.198,0.064
0.35s,3.088,2.624,0.464
0.5s,3.391,2.829,0.562
1s,3.906,3.249,0.657
2s,4.434,3.700,0.734
10s,5.110,4.387,0.722
30s,5.247,4.497,0.749


In [17]:
# V1. sent_ts кратен 1000 µs? → это ms-поле эмиссии (не engine-µs last_updated_at).
ms_share = float((lobs["sent_ts"].to_numpy() % 1000 == 0).mean())
print(f"share sent_ts multiple of 1000us: {ms_share:.4f} → "
      f"{'ms emission field (доп. офсет эмиссии ~десятки мс)' if ms_share > 0.99 else 'engine-us field'}")

share sent_ts multiple of 1000us: 1.0000 → ms emission field (доп. офсет эмиссии ~десятки мс)


In [18]:
# V2 (ГЕЙТ, якорь = timestamp). Детектор просадки задетого уровня; лаг батча-с-импактом от якоря.
# При якоре на матч (g≥0) импакт не протекает в m₀ по построению → доля отрицательных лагов ≈ 0.
ask_px = lobs[[f"ask_px_{k}" for k in range(1, 31)]].to_numpy()
ask_sz = lobs[[f"ask_sz_{k}" for k in range(1, 31)]].to_numpy()
bid_px = lobs[[f"bid_px_{k}" for k in range(1, 31)]].to_numpy()
bid_sz = lobs[[f"bid_sz_{k}" for k in range(1, 31)]].to_numpy()


def size_at_price(idx, price, buy):
    px, sz = (ask_px[idx], ask_sz[idx]) if buy else (bid_px[idx], bid_sz[idx])
    return sz[np.isclose(px, price, atol=TICK / 2)].sum()


def impact_lags_ms(sample, anchor_col):
    """Лаг (мс) батча, где задетый уровень просел на ≥0.5 размера сделки, относительно anchor_col."""
    lags = []
    for _, ev in sample.iterrows():
        price, buy, trade_sz = ev["px_mean"], bool(ev["buy"]), ev["notional"] / ev["px_mean"]
        center = np.searchsorted(book_ts, ev[anchor_col], side="left") - 1
        if center < 3:
            continue
        baseline = size_at_price(center - 3, price, buy)
        for i in range(center - 3, min(center + 8, len(book_ts))):
            if size_at_price(i, price, buy) <= baseline - 0.5 * trade_sz:
                lags.append((book_ts[i] - ev[anchor_col]) / 1000)
                break
    return np.array(lags)


sample = (events[(events["n_levels"] == 1) & (events["notional"] > events["notional"].median())]
          .sort_values("notional", ascending=False).head(500))
lags_hi = impact_lags_ms(sample, "t_match")
print(f"[GATE anchor=timestamp] drops {len(lags_hi)}/{len(sample)} | "
      f"negative-lag share: {float((lags_hi < 0).mean()):.3f} | "
      f"p05={np.percentile(lags_hi, 5):.1f} p50={np.percentile(lags_hi, 50):.1f} p95={np.percentile(lags_hi, 95):.1f}")

gate_hist = go.Figure(go.Histogram(x=lags_hi, nbinsx=60, marker_color="#4C78A8"))
gate_hist.add_vline(x=0, line=dict(color="#E45756", width=1))
gate_hist.update_xaxes(title_text="impact-batch sent_ts − timestamp, ms")
gate_hist.update_layout(title_text="V2 gate: impact-batch lag (anchor = timestamp)")
style_figure(gate_hist, 380).show()

[GATE anchor=timestamp] drops 132/152 | negative-lag share: 0.083 | p05=-81.2 p50=106.5 p95=279.4


In [19]:
# V2 диагностика (старый якорь = transaction_time) + измерение g. negative_lag_share_commit → QA-строка.
lags_lo = impact_lags_ms(sample, "t_commit")
negative_lag_share_commit = float((lags_lo < 0).mean())
is_trade = trades["type"].eq("trade")
g_ms = (trades.loc[is_trade, "transaction_time"] - trades.loc[is_trade, "timestamp"]).to_numpy() / 1000

print(f"[diagnostic anchor=transaction_time] negative-lag share: {negative_lag_share_commit:.3f}")
print(f"g = transaction_time − timestamp, ms:  min={g_ms.min():.1f} p50={np.percentile(g_ms, 50):.1f} "
      f"p95={np.percentile(g_ms, 95):.1f} p99={np.percentile(g_ms, 99):.1f} max={g_ms.max():.1f}")

[diagnostic anchor=transaction_time] negative-lag share: 0.134
g = transaction_time − timestamp, ms:  min=1.0 p50=106.2 p95=266.7 p99=401.4 max=492.5


In [20]:
# V3. Знаковая проверка руками (якорь timestamp): после покупки (side=+1) рост mid → drift > 0.
h035 = HORIZONS_S.index(0.35)
ih_035 = np.clip(np.searchsorted(book_ts, events["t_match"].to_numpy() + int(0.35e6), side="right") - 1,
                 0, len(book_ts) - 1)
sign_check = events.assign(mid0=mid0, mid_h035=book_mid[ih_035], drift_035_bps=drift_hi[:, h035])[valid0_hi]
sign_check[["side", "mid0", "mid_h035", "drift_035_bps", "notional"]].head(8)

,side,mid0,mid_h035,drift_035_bps,notional
order_id,,,,,
21955048350357248,-1,322.1480,322.0065,4.392391,652.623055
21955048350359998,-1,322.0295,322.0295,-0.000000,12.232466
21955048350370682,-1,322.4640,322.4110,1.643594,655.887708
21955048350374758,-1,322.8370,322.8365,0.015488,0.968082
21955048350376935,-1,323.5015,323.3755,3.894881,1596.477435
21955048350377295,-1,322.9935,322.9935,-0.000000,78.480495
21955048350377324,-1,322.9655,322.8745,2.817638,1164.611790
21955048350378072,-1,323.0540,323.0540,-0.000000,37.460344


In [21]:
# Минимальный B_row: AS350 (обе ветки вилки) + time-weighted полуспред p50 + edge + QA.
# Тяжёлая агрегация (bootstrap CI, notional-weighted, buy/sell/liq) отложена до «символ прошёл».
def time_weighted_p50(values, times):
    dwell = np.clip(np.diff(times, append=times[-1]), 0, None)   # время до следующего снапшота
    order = np.argsort(values)
    cw = np.cumsum(dwell[order])
    return float(values[order][np.searchsorted(cw, cw[-1] / 2)])


halfspread_p50_tw = time_weighted_p50(spread["bps"].to_numpy() / 2, spread["recv_ts"].to_numpy())
h_col = {h: k for k, h in enumerate(HORIZONS_S)}
as350_hi = float(np.nanmean(drift_hi[:, h_col[0.35]]))
as350_lo = float(np.nanmean(drift_lo[:, h_col[0.35]]))

B_row = pd.Series({
    "ticker": TICKER,
    "as350_hi_bps": round(as350_hi, 3),
    "as350_lo_bps": round(as350_lo, 3),
    "fork_delta350_bps": round(as350_hi - as350_lo, 3),
    "as1s_hi_bps": round(float(np.nanmean(drift_hi[:, h_col[1]])), 3),
    "as10s_hi_bps": round(float(np.nanmean(drift_hi[:, h_col[10]])), 3),
    "halfspread_p50_tw_bps": round(halfspread_p50_tw, 3),
    "edge_bps": round(halfspread_p50_tw - as350_hi, 3),
    "n_events": int(valid0_hi.sum()),
    "valid_m0_share": round(float(valid0_hi.mean()), 4),
    "negative_lag_share_commit": round(negative_lag_share_commit, 3),
}, name=TICKER)
B_row

ticker                          XMR
as350_hi_bps                  3.088
as350_lo_bps                  2.624
fork_delta350_bps             0.464
as1s_hi_bps                   3.906
as10s_hi_bps                   5.11
halfspread_p50_tw_bps         4.082
edge_bps                      0.994
n_events                        376
valid_m0_share               0.8931
negative_lag_share_commit     0.134
Name: XMR, dtype: object

### 3. Trades analysis

In [22]:
# Поток сделок в $: notional и сторона агрессора. Окно наблюдения — из LOB.
observation_days = (lobs["recv_ts"].iloc[-1] - lobs["recv_ts"].iloc[0]) / 1e6 / 86400

fills = pd.DataFrame({
    "ts": trades["timestamp"].to_numpy(),
    "notional": (trades["price"] * trades["size"]).to_numpy(),
    "taker_buy": trades["is_maker_ask"].eq(True).to_numpy(),  # maker на ask → агрессор покупает
})

print(f"{len(fills):,} fills over {observation_days:.3f} days | "
      f"${fills['notional'].sum():,.0f} total notional")
fills["notional"].describe(percentiles=[.5, .9]).round(2)

510 fills over 0.811 days | $242,729 total notional


count     510.00
mean      475.94
std       597.90
min         0.33
50%       278.50
90%       999.54
max      6660.64
Name: notional, dtype: float64

In [23]:
# Распределение размера сделки в $. Линии p50 (сплошная) и p90 (пунктир).
size_hist = go.Figure(go.Histogram(x=fills["notional"], nbinsx=80, marker_color="#4C78A8"))
for level, dash in ((50, "solid"), (90, "dot")):
    size_hist.add_vline(x=float(np.percentile(fills["notional"], level)),
                        line=dict(color="#E45756", dash=dash, width=1))
size_hist.update_xaxes(range=[0, float(np.percentile(fills["notional"], 99))], title_text="trade notional, $")
size_hist.update_layout(title_text="Trade size distribution, $")
style_figure(size_hist, 380).show()

In [24]:
# Объём по сессии: суммарный notional в $ по бинам.
VOLUME_BIN_MINUTES = 5
session_start = int(fills["ts"].min())
volume_bin = ((fills["ts"] - session_start) // (VOLUME_BIN_MINUTES * 60_000_000)).astype(int)
notional_per_bin = fills.groupby(volume_bin)["notional"].sum()

volume_fig = go.Figure(go.Bar(x=notional_per_bin.index * VOLUME_BIN_MINUTES,
                              y=notional_per_bin.to_numpy(), marker_color="#4C78A8"))
volume_fig.update_xaxes(title_text="minutes from first trade")
volume_fig.update_yaxes(title_text=f"notional per {VOLUME_BIN_MINUTES}-min bin, $")
volume_fig.update_layout(title_text="Traded volume over session")
style_figure(volume_fig, 380).show()

In [25]:
# Строка-артефакт C для сводной таблицы по всем символам.
notional = fills["notional"]
taker_buy = fills["taker_buy"]

C_row = pd.Series({
    "ticker": TICKER,
    "n_trades": len(fills),
    "trades_per_day": round(len(fills) / observation_days, 1),
    "base_volume_total": round(float(trades["size"].sum()), 2),
    "notional_total_usd": round(float(notional.sum()), 2),
    "notional_per_day_usd": round(float(notional.sum()) / observation_days, 2),
    **{f"trade_usd_{name}": value for name, value in percentile_summary(notional, 2).items()},
    "taker_buy_share_count": round(float(taker_buy.mean()), 4),
    "taker_buy_share_notional": round(float(notional[taker_buy].sum() / notional.sum()), 4),
}, name=TICKER)
C_row

ticker                            XMR
n_trades                          510
trades_per_day                  628.8
base_volume_total              741.85
notional_total_usd          242729.29
notional_per_day_usd        299282.49
trade_usd_p1                     4.88
trade_usd_p5                    11.66
trade_usd_p25                   59.26
trade_usd_p50                   278.5
trade_usd_p75                  680.36
trade_usd_p90                  999.54
trade_usd_p95                 1558.61
trade_usd_p99                 2238.42
trade_usd_mean                 475.94
trade_usd_max                 6660.64
taker_buy_share_count          0.4176
taker_buy_share_notional       0.4337
Name: XMR, dtype: object

### 4. Sweep share

In [26]:
# Реконструкция тейкер-ордеров (общий хелпер, см. секцию 0) + доля ликвидаций.
taker_fills, orders = build_taker_orders(trades)
all_notional = (trades["price"] * trades["size"]).to_numpy()
liq_share_notional = 1 - taker_fills["notional"].sum() / all_notional.sum()

print(f"{len(orders):,} taker orders | liq share (notional): {liq_share_notional:.4f} | "
      f"max block_span: {orders['block_span'].max()}")
orders["category"].value_counts()

421 taker orders | liq share (notional): 0.0011 | max block_span: 0


category
single_level    361
sweep            60
Name: count, dtype: int64

In [27]:
# Верхняя граница против дробления: агрессор бьёт серией разных ордеров в одном блоке.
acct_block = taker_fills.groupby(["account", "block"]).agg(
    notional=("notional", "sum"),
    n_levels=("price", "nunique"),
)
sweep_event = acct_block["n_levels"] >= 2
sweep_share_notional_acct_block = float(acct_block.loc[sweep_event, "notional"].sum() / acct_block["notional"].sum())

print(f"sweep share (notional), by (account, block): {sweep_share_notional_acct_block:.4f}")

sweep share (notional), by (account, block): 0.5110


In [28]:
# Распределение глубины свипа в bps (бимодальность: ниблы vs грузовики).
sweep_depth = orders.loc[orders["category"] == "sweep", "depth_bps"]
if len(sweep_depth):
    depth_hist = go.Figure(go.Histogram(x=sweep_depth, nbinsx=80, marker_color="#4C78A8"))
    for level, dash in ((50, "solid"), (90, "dot")):
        depth_hist.add_vline(x=float(np.percentile(sweep_depth, level)),
                             line=dict(color="#E45756", dash=dash, width=1))
    depth_hist.update_xaxes(range=[0, float(np.percentile(sweep_depth, 99))], title_text="sweep depth, bps")
    depth_hist.update_layout(title_text="Sweep depth distribution, bps")
    style_figure(depth_hist, 380).show()
else:
    print("no sweeps to plot")

In [29]:
# Строка-артефакт D для сводной таблицы по всем символам.
total_notional = orders["notional"].sum()
is_sweep = orders["category"] == "sweep"
share_by_category = orders.groupby("category")["notional"].sum() / total_notional
sweeps = orders[is_sweep]


def side_sweep_share(is_buy):
    side = orders["buy"] == is_buy
    side_notional = orders.loc[side, "notional"].sum()
    return float(orders.loc[side & is_sweep, "notional"].sum() / side_notional) if side_notional else np.nan


def pct(series, level):
    return round(float(np.percentile(series, level)), 2) if len(series) else np.nan


D_row = pd.Series({
    "ticker": TICKER,
    "n_taker_orders": len(orders),
    "sweep_share_notional": round(float(share_by_category.get("sweep", 0.0)), 4),
    "sweep_share_count": round(float(is_sweep.mean()), 4),
    "sweep_share_notional_acct_block": round(sweep_share_notional_acct_block, 4),
    "algo_twap_share_notional": round(float(share_by_category.get("algo_twap", 0.0)), 4),
    "liq_share_notional": round(float(liq_share_notional), 4),
    "sweep_share_notional_buy": round(side_sweep_share(True), 4),
    "sweep_share_notional_sell": round(side_sweep_share(False), 4),
    "sweep_depth_bps_p50": pct(sweeps["depth_bps"], 50),
    "sweep_depth_bps_p90": pct(sweeps["depth_bps"], 90),
    "sweep_levels_p50": pct(sweeps["n_levels"], 50),
    "sweep_levels_p90": pct(sweeps["n_levels"], 90),
    "legs_p50": pct(orders["n_legs"], 50),
    "legs_p90": pct(orders["n_legs"], 90),
    "legs_p99": pct(orders["n_legs"], 99),
    "legs_max": int(orders["n_legs"].max()) if len(orders) else 0,
}, name=TICKER)
D_row

ticker                                XMR
n_taker_orders                        421
sweep_share_notional               0.4825
sweep_share_count                  0.1425
sweep_share_notional_acct_block     0.511
algo_twap_share_notional              0.0
liq_share_notional                 0.0011
sweep_share_notional_buy           0.6014
sweep_share_notional_sell          0.3913
sweep_depth_bps_p50                   2.1
sweep_depth_bps_p90                  4.71
sweep_levels_p50                      2.0
sweep_levels_p90                      3.0
legs_p50                              1.0
legs_p90                              2.0
legs_p99                              3.0
legs_max                                5
Name: XMR, dtype: object

### 5. Jump frequency

In [30]:
# Рывки: размах mid внутри неперекрывающихся 1-с бинов (bps). Бин ≤1 с, поэтому >10-с гэп
# в него не попадает → ложных прыжков от разрывов записи нет. Размах (а не endpoint) ловит
# и спайки, которые сходили и вернулись — они всё равно выносят котировку.
JUMP_BPS = 20
jump_ts = book_ts[book_valid]
jump_mid = book_mid[book_valid]
second_bin = (jump_ts - jump_ts[0]) // 1_000_000
per_second = pd.DataFrame({"second": second_bin, "mid": jump_mid}).groupby("second")["mid"].agg(["min", "max"])
per_second["range_bps"] = (per_second["max"] - per_second["min"]) / ((per_second["max"] + per_second["min"]) / 2) * 1e4

jumps_per_day = {thr: round(float((per_second["range_bps"] > thr).sum()) / observation_days, 2) for thr in (20, 50)}
print(f"seconds observed: {len(per_second):,} | jumps/day >20bps: {jumps_per_day[20]} | "
      f">50bps: {jumps_per_day[50]} | max 1s move: {per_second['range_bps'].max():.1f} bps")
per_second["range_bps"].describe(percentiles=[.5, .9, .99]).round(3)

seconds observed: 57,565 | jumps/day >20bps: 4.93 | >50bps: 0.0 | max 1s move: 37.1 bps


count    57565.000
mean         0.179
std          0.726
min          0.000
50%          0.000
90%          0.336
99%          3.482
max         37.090
Name: range_bps, dtype: float64

In [31]:
# Распределение 1-с размахов mid (bps); хвост — источник «каток». Линия порога 20 bps.
jump_hist = go.Figure(go.Histogram(x=per_second["range_bps"], nbinsx=80, marker_color="#4C78A8"))
jump_hist.add_vline(x=JUMP_BPS, line=dict(color="#E45756", width=1))
jump_hist.update_xaxes(range=[0, float(np.percentile(per_second["range_bps"], 99.5))], title_text="1s mid range, bps")
jump_hist.update_layout(title_text="1-second mid range distribution, bps")
style_figure(jump_hist, 380).show()

In [32]:
# Строка-артефакт E для сводной таблицы (только частота рывков; OI/объём отложен).
E_row = pd.Series({
    "ticker": TICKER,
    "obs_days": round(observation_days, 3),
    "seconds_observed": int(len(per_second)),
    "jumps_gt20bps_per_day": jumps_per_day[20],
    "jumps_gt50bps_per_day": jumps_per_day[50],
    "max_1s_move_bps": round(float(per_second["range_bps"].max()), 2),
    "range_bps_p99": round(float(np.percentile(per_second["range_bps"], 99)), 3),
}, name=TICKER)
E_row

ticker                     XMR
obs_days                 0.811
seconds_observed         57565
jumps_gt20bps_per_day     4.93
jumps_gt50bps_per_day      0.0
max_1s_move_bps          37.09
range_bps_p99            3.482
Name: XMR, dtype: object

### 6. Competition

In [33]:
# Конкуренты = аккаунты-мейкеры. Системные id (~2^48: ликвидатор/страховой фонд) выносим отдельно.
SYSTEM_ID = 1_000_000_000_000  # порог «системного» аккаунта — UNVERIFIED, реальные юзеры << этого
is_trade = trades["type"].eq("trade").to_numpy()
maker_ask = trades["is_maker_ask"].to_numpy()
trade_notional = (trades["price"] * trades["size"]).to_numpy()

flow = pd.DataFrame({
    "maker_acct": np.where(maker_ask, trades["ask_account_id"], trades["bid_account_id"]),
    "taker_acct": np.where(maker_ask, trades["bid_account_id"], trades["ask_account_id"]),
    "notional": trade_notional,
})[is_trade]

maker_vol = flow.groupby("maker_acct")["notional"].sum()
system_maker_share = float(maker_vol[maker_vol.index >= SYSTEM_ID].sum() / maker_vol.sum())
user_makers = maker_vol[maker_vol.index < SYSTEM_ID].sort_values(ascending=False)
shares = user_makers / user_makers.sum()

# «Чистота» топ-1: его maker-доля в собственном обороте (MM ли это, а не directional-игрок).
taker_vol = flow.groupby("taker_acct")["notional"].sum()
top1_acct = user_makers.index[0]
top1_purity = float(user_makers.iloc[0] / (user_makers.iloc[0] + taker_vol.get(top1_acct, 0.0)))

print(f"user makers: {len(user_makers):,} | top1 share: {shares.iloc[0]:.3f} | top3: {shares.iloc[:3].sum():.3f} "
      f"| HHI: {(shares ** 2).sum():.4f} | top1 purity: {top1_purity:.3f} | system maker share: {system_maker_share:.3f}")
shares.head(10).round(4)

user makers: 16 | top1 share: 0.656 | top3: 0.929 | HHI: 0.4695 | top1 purity: 1.000 | system maker share: 0.400


maker_acct
418682    0.6562
314236    0.1572
726722    0.1157
283085    0.0185
636906    0.0121
729944    0.0117
724142    0.0096
729694    0.0060
11927     0.0039
556269    0.0034
Name: notional, dtype: float64

In [34]:
# Premium-fee сигнатура: разбивка maker-объёма по тирам maker_fee; доля лучшего (минимального) тира.
# maker_fee — сырой int, единицы/семантика UNVERIFIED → это сигнатура тира, не точная экономика.
maker_fee = trades.loc[is_trade, "maker_fee"].to_numpy()
maker_notional_v = trade_notional[is_trade]
fee_by_tier = pd.Series(maker_notional_v).groupby(maker_fee).sum().sort_index()
best_tier = fee_by_tier.index.min()
premium_tier_share = float(fee_by_tier.loc[best_tier] / fee_by_tier.sum())

print(f"best (min) fee tier {best_tier}: {premium_tier_share:.4f} of maker volume | n tiers: {len(fee_by_tier)}")
(fee_by_tier / fee_by_tier.sum()).round(4)

best (min) fee tier 0: 0.0194 of maker volume | n tiers: 7


0     0.0194
28    0.2483
30    0.0023
32    0.1306
36    0.3937
40    0.0299
50    0.1758
dtype: float64

In [35]:
# Реакция тача (лёгкий прокси): интервал между сменами best_bid/ask, секунды. recv_ts монотонен.
ask1, bid1, rt = lobs["ask_px_1"].to_numpy(), lobs["bid_px_1"].to_numpy(), lobs["recv_ts"].to_numpy()
ok = (ask1 > 0) & (bid1 > 0) & (ask1 >= bid1)
va, vb, vt = ask1[ok], bid1[ok], rt[ok]
touch_changed = (va[1:] != va[:-1]) | (vb[1:] != vb[:-1])
requote_s = np.diff(vt[1:][touch_changed]) / 1e6   # интервалы между сменами тача

print(f"L1 touch changes: {int(touch_changed.sum()):,} | requote interval, s  "
      f"p50={np.percentile(requote_s, 50):.3f}  p90={np.percentile(requote_s, 90):.3f}  "
      f"p99={np.percentile(requote_s, 99):.3f}")

L1 touch changes: 61,262 | requote interval, s  p50=0.054  p90=2.298  p99=19.501


In [36]:
# Доли топ-15 мейкеров в maker-объёме — виден доминант, если есть.
top15 = shares.head(15)
maker_bar = go.Figure(go.Bar(x=[str(a) for a in top15.index], y=top15.to_numpy(), marker_color="#4C78A8"))
maker_bar.update_xaxes(title_text="maker account", type="category")
maker_bar.update_yaxes(title_text="share of maker volume")
maker_bar.update_layout(title_text="Top-15 makers by volume share")
style_figure(maker_bar, 380).show()

In [37]:
# Строка-артефакт F для сводной таблицы.
F_row = pd.Series({
    "ticker": TICKER,
    "n_makers": int(len(user_makers)),
    "n_makers_gt1pct": int((shares > 0.01).sum()),
    "top1_maker_share": round(float(shares.iloc[0]), 4),
    "top3_maker_share": round(float(shares.iloc[:3].sum()), 4),
    "maker_hhi": round(float((shares ** 2).sum()), 4),
    "top1_maker_purity": round(float(top1_purity), 4),
    "system_maker_share": round(float(system_maker_share), 4),
    "premium_tier_share": round(float(premium_tier_share), 4),   # UNVERIFIED units
    "n_fee_tiers": int(len(fee_by_tier)),
    "l1_requote_p50_s": round(float(np.percentile(requote_s, 50)), 3),
    "l1_requote_p90_s": round(float(np.percentile(requote_s, 90)), 3),
}, name=TICKER)
F_row

ticker                   XMR
n_makers                  16
n_makers_gt1pct            6
top1_maker_share      0.6562
top3_maker_share      0.9292
maker_hhi             0.4695
top1_maker_purity        1.0
system_maker_share    0.4001
premium_tier_share    0.0194
n_fee_tiers                7
l1_requote_p50_s       0.054
l1_requote_p90_s       2.298
Name: XMR, dtype: object

### 7. Summary

In [38]:
# Сборка всех артефактов в одну плоскую строку (префикс по критерию, ticker один раз).
criteria_rows = {"A": A_row, "B": B_row, "C": C_row, "D": D_row, "E": E_row, "F": F_row}
summary = {"ticker": TICKER, "ref_price": round(ref_price, 4), "book_gaps": int(len(gap_at))}
for letter, row in criteria_rows.items():
    for name, value in row.items():
        if name != "ticker":
            summary[f"{letter}_{name}"] = value
summary_row = pd.Series(summary, name=TICKER)
summary_row

ticker                      XMR
ref_price               327.517
book_gaps                     0
A_tick                    0.001
A_bps_per_tick           0.0305
                         ...   
F_system_maker_share     0.4001
F_premium_tier_share     0.0194
F_n_fee_tiers                 7
F_l1_requote_p50_s        0.054
F_l1_requote_p90_s        2.298
Name: XMR, Length: 89, dtype: object

In [39]:
# Апсерт строки символа в две общие таблицы: своя строка перезаписывается либо добавляется.
OUT_DIR = ROOT / "research/exchanges/lighter"
SHORT_FIELDS = [
    "ticker", "ref_price", "C_n_trades", "E_obs_days",
    "A_spread_bps_p50", "A_spread_ticks_p50",
    "B_as350_hi_bps", "B_edge_bps",
    "C_trades_per_day", "C_notional_per_day_usd",
    "D_sweep_share_notional",
    "E_jumps_gt20bps_per_day",
    "F_top1_maker_share", "F_l1_requote_p50_s",
]
summary_row_short = summary_row[[k for k in summary_row.index if k in SHORT_FIELDS]]


def upsert_csv(path, row):
    row_df = row.to_frame().T
    if path.exists():
        table = pd.read_csv(path)
        table = table[table["ticker"] != row["ticker"]]
        table = pd.concat([table, row_df], ignore_index=True)
    else:
        table = row_df
    table.sort_values("ticker").reset_index(drop=True).to_csv(path, index=False)


upsert_csv(OUT_DIR / "summary_rows.csv", summary_row)
upsert_csv(OUT_DIR / "summary_rows_short.csv", summary_row_short)
print(f"upserted {TICKER}: summary_rows.csv ({len(summary_row)} cols) + "
      f"summary_rows_short.csv ({len(summary_row_short)} cols) -> {OUT_DIR}")
summary_row_short

upserted XMR: summary_rows.csv (89 cols) + summary_rows_short.csv (14 cols) -> /Users/stepan/Desktop/backtesting/research/exchanges/lighter


ticker                           XMR
ref_price                    327.517
A_spread_ticks_p50             270.0
A_spread_bps_p50               8.259
B_as350_hi_bps                 3.088
B_edge_bps                     0.994
C_n_trades                       510
C_trades_per_day               628.8
C_notional_per_day_usd     299282.49
D_sweep_share_notional        0.4825
E_obs_days                     0.811
E_jumps_gt20bps_per_day         4.93
F_top1_maker_share            0.6562
F_l1_requote_p50_s             0.054
Name: XMR, dtype: object